<a href="https://colab.research.google.com/github/robertbarcik/genai-in-python-tutorial/blob/main/2_basic_project_examples/4_extracting_knowledge_from_videos/4_extracting_knowledge_from_videos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extracting Knowledge from Videos

Hand the app a recording; it **transcribes** the speech and **turns** it into clean, usable notes. The same call works on a lecture, a meeting, or a podcast.

## Setup

Install the library and connect the client to your API key.

In [1]:
%pip install -q openai==2.28.0

import os
from openai import OpenAI

# Key from Colab secrets, an environment variable, or a prompt, in that order.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from getpass import getpass
    api_key = getpass("OpenAI API key: ")

client = OpenAI(api_key=api_key)
TRANSCRIBE_MODEL = "gpt-4o-mini-transcribe"
TEXT_MODEL = "gpt-5-nano"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: /Users/robertbarcik/git-repos/ADK-tutorial/.venv/bin/python3.12 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## The clip you use

A short sample clip (a mock lecture snippet, filler words and all) is bundled here so this runs out of the box. Point `AUDIO_PATH` at your own audio or video file to use it on real content.

In [2]:
from IPython.display import Audio, display

AUDIO_PATH = "sample_lecture_clip.mp3"
display(Audio(AUDIO_PATH))

## The one call

This is the whole engine: send the audio, get the spoken words back as text.

In [3]:
with open(AUDIO_PATH, "rb") as f:
    transcript = client.audio.transcriptions.create(
        model=TRANSCRIBE_MODEL,
        file=f,
    ).text

print(transcript)

Okay, so today I wanna talk about, you know, how spaced repetition works for learning. Basically, the idea is you review material right before you're about to forget it. So like, instead of cramming everything the night before, you spread your reviews out over days and weeks. And every time you successfully recall something, the gap before the next review gets longer. This is kind of the secret behind apps like Anki and Duolingo. So yeah, that's spaced repetition in a nutshell.


## Turn it into notes

Send the raw transcript to a text model and ask for something a lecturer could actually use.

In [4]:
notes = client.responses.create(
    model=TEXT_MODEL,
    input=[
        {"role": "developer", "content":
            "Clean this raw speech transcript into short, clear speaking notes. "
            "Remove filler words, fix punctuation, keep the meaning exactly the same."},
        {"role": "user", "content": transcript},
    ],
).output_text

print(notes)

- Today I want to talk about how spaced repetition works for learning.
- Review material right before you forget it.
- Don’t cram; spread reviews over days and weeks.
- Each successful recall lengthens the interval before the next review.
- This is the secret behind apps like Anki and Duolingo.
- Spaced repetition in a nutshell.


## Take it further

One clip already gets you a transcript and clean notes. A few directions to riff on (good whiteboard fodder, and the API call barely changes):

- **Summarize instead of clean.** Swap the developer prompt for "summarize this in 3 bullet points" for a quick digest.
- **Batch a folder.** Loop the two calls above over every recording in a folder and save each result to a `.txt`.
- **Video files.** Extract the audio track first (e.g. with `moviepy`), then feed it into the same transcription call.

Pick one, sketch the pseudocode, and notice how little new code it needs.

### Your turn

Swap in your own audio or video file for `AUDIO_PATH` and run the last two cells again.